In [14]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [42]:
path = rf"C:\Users\mique\OneDrive\Dokumenty\GitHub\POS\Gesture Emotion Prediction\Results\Patient_1\Q1_1\vid_crop.csv"
df = pd.read_csv(path)
df.columns

Index(['frame', ' face_id', ' timestamp', ' confidence', ' success', ' AU01_r',
       ' AU02_r', ' AU04_r', ' AU05_r', ' AU06_r', ' AU07_r', ' AU09_r',
       ' AU10_r', ' AU12_r', ' AU14_r', ' AU15_r', ' AU17_r', ' AU20_r',
       ' AU23_r', ' AU25_r', ' AU26_r', ' AU45_r', ' AU01_c', ' AU02_c',
       ' AU04_c', ' AU05_c', ' AU06_c', ' AU07_c', ' AU09_c', ' AU10_c',
       ' AU12_c', ' AU14_c', ' AU15_c', ' AU17_c', ' AU20_c', ' AU23_c',
       ' AU25_c', ' AU26_c', ' AU28_c', ' AU45_c'],
      dtype='object')

In [29]:
def extract_features(csv_path):

    df = pd.read_csv(csv_path)

    # keep only valid frames
    if "success" in df.columns:
        df = df[df["success"] == 1]

    # select AU intensity columns
    au_cols = [c for c in df.columns if "_r" in c]

    features = {}

    for col in au_cols:
        features[f"{col}_mean"] = df[col].mean()
        features[f"{col}_std"] = df[col].std()
        features[f"{col}_max"] = df[col].max()

    return features

In [30]:
quadrants_all = [
    "Q1_1",
    "Q1_2",
    "Q2_1",
    "Q2_2",
    "Q3_1",
    "Q3_2",
    "Q4_1",
    "Q4_2",
    "Q5_1",
    "Q5_2",
    "Q6_1",
    "Q6_2",
    "Q7_1",
    "Q7_2",
    "Q8_1",
    "Q8_2",
    "Q9_1",
    "Q9_2"
]

quadrants_arousal = [
    "Q1_1",
    "Q1_2",
    "Q2_1",
    "Q2_2",
    "Q3_1",
    "Q3_2",
    "Q7_1",
    "Q7_2",
    "Q8_1",
    "Q8_2",
    "Q9_1",
    "Q9_2"
]

quadrants_valence = [
    "Q1_1",
    "Q1_2",
    "Q3_1",
    "Q3_2",
    "Q4_1",
    "Q4_2",
    "Q6_1",
    "Q6_2",
    "Q7_1",
    "Q7_2",
    "Q9_1",
    "Q9_2"
]

quadrants_5q = [
    "Q1_1",
    "Q1_2",
    "Q3_1",
    "Q3_2",
    "Q5_1",
    "Q5_2",
    "Q7_1",
    "Q7_2",
    "Q9_1",
    "Q9_2"
]

patients = list(range(1, 62))

patients.remove(23)

#print(patients)

In [31]:
quadrants = quadrants_5q

In [32]:
def get_label(quadrant):
    return quadrant.split("_")[0]

In [33]:
# def get_label(quadrant):
#     q = int(quadrant[1])
#     if q < 4:
#         return "High Arousal"
#     else:
#         return "Low Arousal"

In [34]:
# def get_label(quadrant):
#     q = int(quadrant[1])
#     if q in [1, 4, 7]:
#         return "Negative Valence"
#     else:
#         return "Positive Valence"

In [35]:
data = []
labels = []

for patient in patients:

    patient = f"Patient_{patient}"

    print(f"Computing Features for {patient}...", end="\r", flush=True)

    for idx, quadrant in enumerate(quadrants):

        path = rf"C:\Users\mique\OneDrive\Dokumenty\GitHub\POS\Gesture Emotion Prediction\Results\{patient}\{quadrant}\vid_crop.csv"

        features = extract_features(path)

        # example label extraction
        label = get_label(quadrant)   # Q1, Q2, Q3...

        data.append(features)
        labels.append(label)

X = pd.DataFrame(data)
y = np.array(labels)


print()
print("Dataset shape:", X.shape)

Computing Features for Patient_61...
Dataset shape: (600, 51)


In [36]:
X.head()

,AU01_r_mean,AU01_r_std,AU01_r_max,AU02_r_mean,AU02_r_std,AU02_r_max,AU04_r_mean,AU04_r_std,AU04_r_max,AU05_r_mean,...,AU23_r_max,AU25_r_mean,AU25_r_std,AU25_r_max,AU26_r_mean,AU26_r_std,AU26_r_max,AU45_r_mean,AU45_r_std,AU45_r_max
0,0.283856,0.488994,2.17,0.187121,0.435430,2.02,0.788090,0.614241,2.58,0.042799,...,0.81,0.264694,0.285527,1.71,0.399410,0.399819,1.95,0.423243,0.624285,2.33
1,0.141139,0.272389,1.71,0.076906,0.145022,0.92,0.585757,0.567543,2.68,0.037078,...,0.55,0.178846,0.153283,0.79,0.255109,0.216361,1.02,0.385251,0.726524,3.12
2,0.426009,0.923254,4.62,0.285135,0.721862,3.50,0.325726,0.256046,1.97,0.162163,...,0.93,0.259047,0.346705,3.19,0.393346,0.484862,2.88,0.321269,0.615180,2.93
3,0.238746,0.442046,2.54,0.137423,0.323328,2.01,0.342772,0.281015,2.22,0.091986,...,1.20,0.282120,0.415636,3.29,0.479984,0.539432,2.91,0.254037,0.424908,2.59
4,0.188612,0.395135,2.96,0.080047,0.208254,2.22,0.352402,0.210107,1.55,0.038833,...,0.53,0.157125,0.108611,0.62,0.171221,0.124028,0.57,0.318753,0.617475,2.64


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [39]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [40]:
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

          Q1       0.37      0.29      0.33        24
          Q3       0.61      0.58      0.60        24
          Q5       0.32      0.33      0.33        24
          Q7       0.32      0.38      0.35        24
          Q9       0.40      0.42      0.41        24

    accuracy                           0.40       120
   macro avg       0.40      0.40      0.40       120
weighted avg       0.40      0.40      0.40       120

[[ 7  6  6  3  2]
 [ 6 14  1  3  0]
 [ 1  0  8  7  8]
 [ 2  3  5  9  5]
 [ 3  0  5  6 10]]


In [41]:
importances = pipe.named_steps["clf"].feature_importances_

feat_imp = pd.Series(importances, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(20))

AU12_r_std     0.034588
AU06_r_std     0.034299
AU10_r_std     0.030210
AU09_r_mean    0.028156
AU09_r_std     0.027217
AU06_r_max     0.025649
AU07_r_std     0.024198
AU04_r_std     0.023129
AU12_r_max     0.022253
AU05_r_max     0.021901
AU04_r_max     0.021475
AU02_r_max     0.020895
AU07_r_max     0.020723
AU09_r_max     0.020658
AU10_r_max     0.020617
AU20_r_max     0.020518
AU25_r_max     0.020509
AU26_r_max     0.020450
AU01_r_max     0.019636
AU25_r_std     0.019390
dtype: float64
